# StereoQueerEval Task B: Pretrained Attention Weight Borrowing with $K$-Latent Queries
### End-to-End Kaggle Notebook (GPU P100 / T4 / A100)

**Key Architectural Highlights:**
- **Context Stream**: `mmBERT` (ModernBERT base, 22 layers, 768 hidden, 12 heads) with native full/sliding-window attention masks.
- **Query Stream**: $K$ general learnable latent query slots $Z_0 \in \mathbb{R}^{B \times K \times 768}$ initialized $\sim \mathcal{N}(0, 0.02)$.
- **Transplanted Cross-Attention Stack (Layers 18 $\to$ 22)**: Parameter-decoupled linear projections with **Asymmetric RoPE** (Keys rotated with native ModernBERT RoPE, Queries position-free).
- **Context Synchronization**: Layer-matched mode ($K_l, V_l$ receive $H_{l-1}$ for $l \in [18..22]$).
- **Hierarchical Tree Classifier**: Attentive readout producing compound probabilities $P(\text{no})$, $P(\text{yes\_implicit})$, $P(\text{yes\_explicit})$.
- **Compound Log-Sigmoid Loss**: Joint NLL via `F.logsigmoid`.

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install modern transformers with native ModernBERT support
!pip install -q "transformers>=4.48.0" accelerate scikit-learn pandas numpy matplotlib seaborn tqdm

import os
import sys
import math
import copy
import json
import random
from dataclasses import dataclass, field
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModel, AutoConfig

# Set seeds for deterministic reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 2. Model Configuration

In [ ]:
@dataclass
class ExperimentConfig:
    model_name: str = "jhu-clsp/mmbert-base"
    num_classes: int = 3
    hidden_dim: int = 768
    num_query_slots: int = 8             # K general latent queries
    split_layer_idx: int = 17            # mmBERT layers 0..16 -> Context Stream
    transplant_layers_count: int = 5     # mmBERT layers 17..21 (Layers 18 to 22)
    context_mode: str = "layer_matched"  # 'layer_matched' (Layer l gets H_{l-1}) vs 'static_h17'
    use_prenorm: bool = False            # False = Exp A (Pure Projections), True = Exp B (Pre-LN)
    use_ffn: bool = False                # False = Projections only (Primary), True = Ablation 4
    freeze_encoder: bool = True          # Freeze Context Layers 1..17
    freeze_transplant: bool = True       # Freeze Transplant Projections (Isolated Hypothesis)
    
    # Training
    max_length: int = 384
    batch_size: int = 16
    epochs: int = 10
    lr: float = 2e-4
    weight_decay: float = 0.01
    val_size: float = 0.15
    output_dir: str = "./outputs_task_b"

cfg = ExperimentConfig()
os.makedirs(cfg.output_dir, exist_ok=True)

## 3. Data Ingestion & Context-Aware Preprocessing

In [ ]:
# Helper to format Comment + Video Title + Description with contextual separators
def format_context_input(comment: str, title: str = "", desc: str = "") -> str:
    parts = [f"Comment: {str(comment).strip()}"]
    if pd.notna(title) and str(title).strip():
        parts.append(f"Video: {str(title).strip()}")
    if pd.notna(desc) and str(desc).strip():
        parts.append(f"Description: {str(desc).strip()[:200]}")
    return " [SEP] ".join(parts)

# Mock / Synthetic Dataset Loader for demonstration if TSV files are in local path
def load_or_create_dataset():
    tsv_paths = [
        "./data/train_task_b_it.tsv", "./data/train_task_b_nl.tsv",
        "./data/train_task_b_en.tsv", "./data/train_task_b_fa.tsv",
        "../input/stereoqueereval-task-b/train_task_b.tsv"
    ]
    dfs = []
    for p in tsv_paths:
        if os.path.exists(p):
            dfs.append(pd.read_csv(p, sep='\t'))
    
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        print(f"Loaded {len(df)} real records from TSVs.")
    else:
        print("No local TSVs found; creating representative multi-lingual synthetic dataset...")
        langs = ['it', 'nl', 'en', 'fa'] * 125
        labels = ['no', 'yes_implicit', 'yes_explicit'] * 166 + ['no', 'yes_implicit']
        comments = [
            f"Sample multi-lingual social media comment {i} discussing sensitive topic" for i in range(500)
        ]
        titles = [f"YouTube Video Discussion #{i % 40}" for i in range(500)]
        descs = [f"Contextual description for video #{i % 40}" for i in range(500)]
        df = pd.DataFrame({
            'id': [f"sample_{i}" for i in range(500)],
            'language': langs,
            'comment': comments,
            'yt_title': titles,
            'yt_description': descs,
            'label': labels
        })
    
    label_map = {'no': 0, 'yes_implicit': 1, 'yes_explicit': 2}
    df['label_id'] = df['label'].map(label_map)
    df['formatted_text'] = df.apply(
        lambda r: format_context_input(r['comment'], r.get('yt_title', ''), r.get('yt_description', '')),
        axis=1
    )
    return df

raw_df = load_or_create_dataset()
print(raw_df[['language', 'label', 'formatted_text']].head(3))

### Leakage-Free Video Group Split

In [ ]:
# Group split by video title
unique_videos = raw_df['yt_title'].dropna().unique()
np.random.shuffle(unique_videos)
n_val = max(1, int(len(unique_videos) * cfg.val_size))
val_videos = set(unique_videos[:n_val])

train_df = raw_df[~raw_df['yt_title'].isin(val_videos)].reset_index(drop=True)
val_df = raw_df[raw_df['yt_title'].isin(val_videos)].reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")
print("Class distribution (Train):", train_df['label'].value_counts().to_dict())

# Balanced class weights for Level 1 and Level 2
classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=train_df['label_id'].values)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
print(f"Balanced Class Weights: {weights.round(3).tolist()}")

## 4. PyTorch Dataset & Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

class TaskBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TaskBDataset(train_df['formatted_text'].values, train_df['label_id'].values, tokenizer, cfg.max_length)
val_dataset = TaskBDataset(val_df['formatted_text'].values, val_df['label_id'].values, tokenizer, cfg.max_length)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size * 2, shuffle=False)

## 5. Model Implementation ($K$-Latent Attention Transplant)

In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb_single(k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    if cos.dim() == 3:
        cos = cos.unsqueeze(1)
        sin = sin.unsqueeze(1)
    elif cos.dim() == 2:
        cos = cos.unsqueeze(0).unsqueeze(1)
        sin = sin.unsqueeze(0).unsqueeze(1)
    return (k * cos) + (rotate_half(k) * sin)

try:
    from transformers.models.modernbert.modeling_modernbert import (
        create_bidirectional_mask,
        create_bidirectional_sliding_window_mask
    )
except ImportError:
    create_bidirectional_mask = None
    create_bidirectional_sliding_window_mask = None

def build_native_attention_masks(config, inputs_embeds, attention_mask=None, sliding_window=None):
    if create_bidirectional_mask is not None and create_bidirectional_sliding_window_mask is not None:
        try:
            return {
                "full_attention": create_bidirectional_mask(config, inputs_embeds, attention_mask),
                "sliding_attention": create_bidirectional_sliding_window_mask(config, inputs_embeds, attention_mask),
                "global_attention": create_bidirectional_mask(config, inputs_embeds, attention_mask),
            }
        except Exception:
            pass
    B, L, _ = inputs_embeds.shape
    device = inputs_embeds.device
    dtype = inputs_embeds.dtype
    min_val = torch.finfo(dtype).min if dtype.is_floating_point else -10000.0
    window_size = sliding_window or getattr(config, "local_attention", 128)
    pad_mask = (1.0 - attention_mask.unsqueeze(1).unsqueeze(2).to(dtype)) * min_val if attention_mask is not None else torch.zeros((B, 1, 1, L), device=device, dtype=dtype)
    row_idx = torch.arange(L, device=device).unsqueeze(1)
    col_idx = torch.arange(L, device=device).unsqueeze(0)
    band = (col_idx >= row_idx - window_size // 2) & (col_idx <= row_idx + window_size // 2)
    band_mask = torch.where(band.unsqueeze(0).unsqueeze(0), 0.0, min_val).to(dtype)
    return {
        "full_attention": pad_mask,
        "sliding_attention": pad_mask + band_mask,
        "global_attention": pad_mask,
    }

class PretrainedCrossAttentionLayer(nn.Module):
    def __init__(self, pretrained_attn, rotary_emb=None, pretrained_norm=None, pretrained_mlp=None, pretrained_mlp_norm=None, layer_type="full_attention", use_prenorm=False, use_ffn=False):
        super().__init__()
        self.use_prenorm = use_prenorm
        self.use_ffn = use_ffn
        self.layer_type = layer_type

        # Cloned weights for decoupled freezing control
        self.Wqkv = copy.deepcopy(pretrained_attn.Wqkv)
        self.Wo = copy.deepcopy(pretrained_attn.Wo)
        self.attn_norm = copy.deepcopy(pretrained_norm) if (use_prenorm and pretrained_norm is not None) else None
        self.mlp = copy.deepcopy(pretrained_mlp) if (use_ffn and pretrained_mlp is not None) else None
        self.mlp_norm = copy.deepcopy(pretrained_mlp_norm) if (use_ffn and use_prenorm and pretrained_mlp_norm is not None) else None
        self.rotary_emb = rotary_emb or getattr(pretrained_attn, "rotary_emb", None)
        self.num_heads = pretrained_attn.config.num_attention_heads
        self.head_dim = getattr(pretrained_attn, "head_dim", pretrained_attn.config.hidden_size // self.num_heads)
        self.scaling = self.head_dim ** -0.5

    def forward(self, z, h_context, context_mask=None, position_ids=None):
        B, K, D = z.shape
        _, L, _ = h_context.shape
        z_in = self.attn_norm(z) if (self.use_prenorm and self.attn_norm is not None) else z
        h_in = self.attn_norm(h_context) if (self.use_prenorm and self.attn_norm is not None) else h_context

        q_weight, k_weight, v_weight = self.Wqkv.weight.chunk(3, dim=0)
        q = F.linear(z_in, q_weight).view(B, K, self.num_heads, self.head_dim).transpose(1, 2)
        k = F.linear(h_in, k_weight).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = F.linear(h_in, v_weight).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        if self.rotary_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(L, device=h_context.device).unsqueeze(0).expand(B, -1)
            try:
                cos, sin = self.rotary_emb(h_context, position_ids, layer_type=self.layer_type)
            except TypeError:
                cos, sin = self.rotary_emb(h_context, position_ids)
            k = apply_rotary_pos_emb_single(k, cos, sin)

        scores = torch.matmul(q, k.transpose(-1, -2)) * self.scaling
        if context_mask is not None:
            mask_4d = (1.0 - context_mask.unsqueeze(1).unsqueeze(2).to(scores.dtype)) * -10000.0 if context_mask.dim() == 2 else context_mask.to(scores.dtype)
            scores = scores + mask_4d

        attn_weights = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, v).transpose(1, 2).contiguous().view(B, K, D)
        z_out = z + self.Wo(attn_out)

        if self.use_ffn and self.mlp is not None:
            z_mlp_in = self.mlp_norm(z_out) if (self.use_prenorm and self.mlp_norm is not None) else z_out
            z_out = z_out + self.mlp(z_mlp_in)
        return z_out, attn_weights.mean(dim=1)

class HierarchicalTreeHead(nn.Module):
    def __init__(self, hidden_dim=768, num_query_slots=8, dropout=0.1):
        super().__init__()
        self.query_hate = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.query_fine = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.val_proj = nn.Linear(hidden_dim, hidden_dim)
        self.fc_hate = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1))
        self.fc_fine = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1))

    def _readout(self, query, z_latents):
        B, K, D = z_latents.shape
        q = query.expand(B, -1, -1)
        k = self.key_proj(z_latents)
        v = self.val_proj(z_latents)
        attn = F.softmax(torch.bmm(q, k.transpose(1, 2)) * (D ** -0.5), dim=-1)
        return torch.bmm(attn, v).squeeze(1)

    def forward(self, z_latents):
        z_h = self._readout(self.query_hate, z_latents)
        z_f = self._readout(self.query_fine, z_latents)
        logit_h = self.fc_hate(z_h).squeeze(-1)
        logit_f = self.fc_fine(z_f).squeeze(-1)
        p_hate = torch.sigmoid(logit_h)
        p_imp = torch.sigmoid(logit_f)
        compound_probs = torch.stack([1.0 - p_hate, p_hate * p_imp, p_hate * (1.0 - p_imp)], dim=-1)
        return logit_h, logit_f, compound_probs

class TransplantedKLatentModel(nn.Module):
    def __init__(self, config: ExperimentConfig):
        super().__init__()
        self.cfg = config
        self.model_config = AutoConfig.from_pretrained(config.model_name)
        self.backbone = AutoModel.from_pretrained(config.model_name, config=self.model_config)
        layers = getattr(self.backbone, "layers", None) or getattr(self.backbone.encoder, "layers", None)
        self.rotary_emb = getattr(self.backbone, "rotary_emb", None) or getattr(layers[0].attn, "rotary_emb", None)

        self.query_probes = nn.Parameter(torch.randn(config.num_query_slots, config.hidden_dim) * 0.02)
        self.transplant_layers = nn.ModuleList()
        for idx in range(config.split_layer_idx, config.split_layer_idx + config.transplant_layers_count):
            layer_module = layers[idx]
            l_type = getattr(layer_module, "attention_type", getattr(layer_module, "layer_type", "full_attention"))
            self.transplant_layers.append(PretrainedCrossAttentionLayer(
                pretrained_attn=layer_module.attn,
                rotary_emb=self.rotary_emb,
                pretrained_norm=getattr(layer_module, "attn_norm", None),
                pretrained_mlp=getattr(layer_module, "mlp", None),
                pretrained_mlp_norm=getattr(layer_module, "mlp_norm", None),
                layer_type=l_type,
                use_prenorm=config.use_prenorm,
                use_ffn=config.use_ffn
            ))

        self.tree_head = HierarchicalTreeHead(hidden_dim=config.hidden_dim, num_query_slots=config.num_query_slots)

        # Freeze enforcement
        for p in self.backbone.parameters():
            p.requires_grad = not config.freeze_encoder
        for layer in self.transplant_layers:
            for p in layer.parameters():
                p.requires_grad = not config.freeze_transplant
        self.query_probes.requires_grad = True
        for p in self.tree_head.parameters():
            p.requires_grad = True

    def encode_context(self, input_ids, attention_mask=None, position_ids=None):
        B, L = input_ids.shape
        layers = getattr(self.backbone, "layers", None) or getattr(self.backbone.encoder, "layers", None)
        if position_ids is None:
            position_ids = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, -1)
        hidden_states = self.backbone.embeddings(input_ids)
        saved = [hidden_states]
        mask_map = build_native_attention_masks(self.model_config, hidden_states, attention_mask)
        layer_types = getattr(self.model_config, "layer_types", ["full_attention", "sliding_attention"])
        pos_embs = {lt: self.rotary_emb(hidden_states, position_ids, lt) if self.rotary_emb else None for lt in set(layer_types)}

        # Execute layers up to 21 (0..20) in layer-matched mode, saving 22nd layer compute
        n_run = self.cfg.split_layer_idx + self.cfg.transplant_layers_count - 1 if self.cfg.context_mode == "layer_matched" else self.cfg.split_layer_idx
        for i in range(n_run):
            l = layers[i]
            at = getattr(l, "attention_type", getattr(l, "layer_type", "full_attention"))
            out = l(hidden_states, attention_mask=mask_map.get(at, mask_map["full_attention"]), position_embeddings=pos_embs.get(at))
            hidden_states = out[0] if isinstance(out, tuple) else out
            saved.append(hidden_states)
        return saved, position_ids

    def forward(self, input_ids, attention_mask, position_ids=None):
        B, L = input_ids.shape
        with torch.set_grad_enabled(not self.cfg.freeze_encoder):
            saved, position_ids = self.encode_context(input_ids, attention_mask, position_ids)
        z = self.query_probes.unsqueeze(0).expand(B, -1, -1)
        last_attn = None
        for i, t_layer in enumerate(self.transplant_layers):
            h_ctx = saved[self.cfg.split_layer_idx + i] if self.cfg.context_mode == "layer_matched" else saved[self.cfg.split_layer_idx]
            z, last_attn = t_layer(z=z, h_context=h_ctx, context_mask=attention_mask, position_ids=position_ids)
        logit_h, logit_f, compound_probs = self.tree_head(z)
        return logit_h, logit_f, compound_probs, last_attn

## 6. Compound Log-Sigmoid Loss Function

In [ ]:
class HierarchicalCompoundLoss(nn.Module):
    def __init__(self, class_weights=None, level1_weight=1.0, level2_weight=1.0):
        super().__init__()
        self.class_weights = class_weights
        self.level1_weight = level1_weight
        self.level2_weight = level2_weight

    def forward(self, logit_h, logit_f, targets):
        mask_no = (targets == 0)
        mask_implicit = (targets == 1)
        mask_explicit = (targets == 2)

        loss_no = -F.logsigmoid(-logit_h) * (self.class_weights[0] if self.class_weights is not None else 1.0)
        loss_imp = -(F.logsigmoid(logit_h) + F.logsigmoid(logit_f)) * (self.class_weights[1] if self.class_weights is not None else 1.0)
        loss_exp = -(F.logsigmoid(logit_h) + F.logsigmoid(-logit_f)) * (self.class_weights[2] if self.class_weights is not None else 1.0)

        total_loss = torch.zeros_like(logit_h)
        total_loss = torch.where(mask_no, loss_no, total_loss)
        total_loss = torch.where(mask_implicit, loss_imp, total_loss)
        total_loss = torch.where(mask_explicit, loss_exp, total_loss)
        return total_loss.mean()

loss_fn = HierarchicalCompoundLoss(class_weights=class_weights_tensor)

## 7. Per-Epoch Training Loop with Comprehensive Logging

In [ ]:
model = TransplantedKLatentModel(cfg).to(device)

# Parameter groups
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable Parameters: {sum(p.numel() for p in trainable_params):,}")

optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

# Training Metrics Log
training_logs = []
best_macro_f1 = -1.0
best_checkpoint_path = os.path.join(cfg.output_dir, "best_task_b_model.pt")

print("\n" + "="*75)
print(f"{'Epoch':^7} | {'Train Loss':^11} | {'Val Acc':^9} | {'Val Macro F1':^13} | {'F1 (no/imp/exp)':^20}")
print("="*75)

for epoch in range(1, cfg.epochs + 1):
    # --- Training Phase ---
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs} [Train]", leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logit_h, logit_f, _, _ = model(input_ids, attention_mask)
        loss = loss_fn(logit_h, logit_f, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * len(labels)

    scheduler.step()
    avg_train_loss = train_loss / len(train_dataset)

    # --- Validation Phase ---
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            _, _, compound_probs, _ = model(input_ids, attention_mask)
            preds = torch.argmax(compound_probs, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # Logging record
    f1_str = f"{per_class_f1[0]:.2f} / {per_class_f1[1]:.2f} / {per_class_f1[2]:.2f}"
    print(f"{epoch:^7} | {avg_train_loss:^11.4f} | {val_acc:^9.4f} | {val_macro_f1:^13.4f} | {f1_str:^20}")

    training_logs.append({
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'val_accuracy': val_acc,
        'val_macro_f1': val_macro_f1,
        'f1_no': per_class_f1[0],
        'f1_implicit': per_class_f1[1],
        'f1_explicit': per_class_f1[2],
        'lr': optimizer.param_groups[0]['lr']
    })

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        torch.save(model.state_dict(), best_checkpoint_path)

print("="*75)
print(f"Training Complete. Best Validation Macro F1: {best_macro_f1:.4f}")

# Save training log to CSV
log_df = pd.DataFrame(training_logs)
log_df.to_csv(os.path.join(cfg.output_dir, "training_history.csv"), index=False)

## 8. Results Analysis & Diagnostic Visualizations

In [ ]:
# Load best model weights
model.load_state_dict(torch.load(best_checkpoint_path))
model.eval()

# Plot Learning Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(log_df['epoch'], log_df['train_loss'], marker='o', color='crimson', label='Train Loss')
ax1.set_title('Hierarchical Compound Loss Curve', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.plot(log_df['epoch'], log_df['val_macro_f1'], marker='s', color='navy', label='Val Macro F1')
ax2.plot(log_df['epoch'], log_df['val_accuracy'], marker='^', color='teal', label='Val Accuracy')
ax2.plot(log_df['epoch'], log_df['f1_implicit'], marker='.', color='orange', linestyle=':', label='F1 (Implicit)')
ax2.set_title('Validation Metrics per Epoch', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "learning_curves.png"), dpi=300)
plt.show()

### Confusion Matrix & Classification Report

In [ ]:
target_names = ['no', 'yes_implicit', 'yes_explicit']
print("\nClassification Report (Validation Set):")
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Normalized Confusion Matrix (Task B)', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "confusion_matrix.png"), dpi=300)
plt.show()

### Cross-Lingual Performance Breakdown

In [ ]:
val_df_analysis = val_df.copy()
val_df_analysis['prediction'] = [target_names[p] for p in all_preds]
val_df_analysis['ground_truth'] = [target_names[l] for l in all_labels]

lang_rows = []
for lang, sub in val_df_analysis.groupby('language'):
    macro_f1 = f1_score(sub['ground_truth'], sub['prediction'], average='macro', zero_division=0)
    acc = accuracy_score(sub['ground_truth'], sub['prediction'])
    lang_rows.append({'Language': lang.upper(), 'Samples': len(sub), 'Macro F1': macro_f1, 'Accuracy': acc})

lang_df = pd.DataFrame(lang_rows)
print("\nPer-Language Evaluation Breakdown:")
display(lang_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=lang_df, x='Language', y='Macro F1', palette='viridis')
plt.title('Task B Macro F1 Across Languages (IT, NL, EN, FA)', fontweight='bold')
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "language_breakdown.png"), dpi=300)
plt.show()

## 9. Inference & Submission Generator

In [ ]:
def generate_submission(test_tsv_path: str, output_tsv_path: str):
    if not os.path.exists(test_tsv_path):
        print(f"Test file {test_tsv_path} not found. Creating placeholder test submission.")
        test_df = val_df[['id', 'formatted_text']].copy()
    else:
        test_df = pd.read_csv(test_tsv_path, sep='\t')
        test_df['formatted_text'] = test_df.apply(
            lambda r: format_context_input(r['comment'], r.get('yt_title', ''), r.get('yt_description', '')),
            axis=1
        )

    test_dataset = TaskBDataset(test_df['formatted_text'].values, None, tokenizer, cfg.max_length)
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size * 2, shuffle=False)

    predictions = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting Test Set"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            _, _, compound_probs, _ = model(input_ids, attention_mask)
            preds = torch.argmax(compound_probs, dim=-1)
            predictions.extend(preds.cpu().numpy())

    test_df['label'] = [target_names[p] for p in predictions]
    sub_df = test_df[['id', 'label']]
    sub_df.to_csv(output_tsv_path, sep='\t', index=False)
    print(f"Submission successfully written to {output_tsv_path} ({len(sub_df)} rows).")
    return sub_df

# Generate submission
sub_path = os.path.join(cfg.output_dir, "submission_task_b.tsv")
sub_df = generate_submission("./data/test_task_b.tsv", sub_path)
print(sub_df.head(5))